# Online Feature Tables

DO NOT RUN THIS NOTEBOOK ON CORPORATE ACCOUNT. IN FREE ACCOUNT, YOU MUST REMOVE TABLES AND MODELS. IT WILL BE COVERED POST AI Topics.

Serve the latest machine-learning feature values by customer key. This standalone notebook creates source data and a view, builds an online feature table, queries entity keys, refreshes changes, and shows an optional Snowflake ML Feature Store path.

Run SQL in Snowsight ie SnowFlake UI portal. Run the optional Python block in a Snowflake Python notebook. No event-table or interactive-table lab is required.

## 1. Set up the schema

Select an existing writable database and authorized role. Run this setup once, using the existing learning warehouse:

```sql
CREATE SCHEMA OnlineFeatures;
USE SCHEMA OnlineFeatures;
USE WAREHOUSE SNOWFLAKE_LEARNING_WH;
SELECT CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE(), CURRENT_ROLE();
```

Replace `OnlineFeatures` consistently with your team's schema name if needed. Keep the same database selected. The role needs source-object privileges, `CREATE ONLINE FEATURE TABLE` on this schema, and warehouse usage. Availability depends on the online-serving configuration and account; hybrid-backed serving has hybrid-table restrictions, including trial-account limitations. Verify eligibility before the serving-table section.

[Online serving prerequisites](https://docs.snowflake.com/en/developer-guide/snowflake-ml/feature-store/online-feature-store)

## 2. Latest features by entity key

An offline feature pipeline creates training and batch-scoring values. Online serving copies the latest feature row for each entity into a low-latency lookup store. The required primary key is the entity lookup key. A timestamp column tells the service how to order feature observations when relevant.

The SQL object can read from a **view or dynamic table**, not directly from an arbitrary table. Target lag must be between 10 seconds and 8 days. Snowflake ML Feature Store can create and manage the same hybrid-backed online serving path with `OnlineConfig`; direct SQL makes the underlying type easy to study. Source: [CREATE ONLINE FEATURE TABLE](https://docs.snowflake.com/en/sql-reference/sql/create-online-feature-table).

## 3. Create source data and a view

Keep one current row per `CUSTOMER_ID`. Production features might be calculated by a dynamic table instead.

```sql
USE SCHEMA OnlineFeatures;

CREATE TABLE CUSTOMER_FEATURE_SOURCE_DATA (
  CUSTOMER_ID INTEGER,
  FEATURE_TS TIMESTAMP_NTZ,
  ORDERS_30D INTEGER,
  SPEND_30D NUMBER(12,2),
  DAYS_SINCE_LAST_ORDER INTEGER,
  RISK_SCORE FLOAT
);

INSERT INTO CUSTOMER_FEATURE_SOURCE_DATA VALUES
  (101, '2026-09-07 09:00:00', 4, 19600.00, 1, 0.08),
  (102, '2026-09-07 09:00:00', 1,  2200.00, 6, 0.31),
  (103, '2026-09-07 09:00:00', 2,  7400.00, 3, 0.16);

CREATE VIEW CUSTOMER_FEATURE_SOURCE AS
SELECT CUSTOMER_ID, FEATURE_TS, ORDERS_30D, SPEND_30D,
       DAYS_SINCE_LAST_ORDER, RISK_SCORE
FROM CUSTOMER_FEATURE_SOURCE_DATA;
```

## 4. Create, query, inspect, and refresh

This requires `CREATE ONLINE FEATURE TABLE` on the schema and `USAGE` on the refresh warehouse. Hybrid-backed online feature serving also follows the account availability constraints described in Snowflake's online feature documentation.

```sql
CREATE ONLINE FEATURE TABLE CUSTOMER_FEATURES_ONLINE
  PRIMARY KEY (CUSTOMER_ID)
  TIMESTAMP_COLUMN = 'FEATURE_TS'
  TARGET_LAG = '30 seconds'
  WAREHOUSE = SNOWFLAKE_LEARNING_WH
  REFRESH_MODE = INCREMENTAL
  COMMENT = 'Latest behavioral features for customer scoring'
FROM CUSTOMER_FEATURE_SOURCE;

SELECT CUSTOMER_ID, ORDERS_30D, SPEND_30D, RISK_SCORE
FROM CUSTOMER_FEATURES_ONLINE
WHERE CUSTOMER_ID IN (101, 103);

SHOW ONLINE FEATURE TABLES LIKE 'CUSTOMER_FEATURES_ONLINE';
DESCRIBE ONLINE FEATURE TABLE CUSTOMER_FEATURES_ONLINE;

UPDATE CUSTOMER_FEATURE_SOURCE_DATA
SET FEATURE_TS = CURRENT_TIMESTAMP(),
    ORDERS_30D = 5,
    SPEND_30D = 22800.00,
    DAYS_SINCE_LAST_ORDER = 0,
    RISK_SCORE = 0.05
WHERE CUSTOMER_ID = 101;

DELETE FROM CUSTOMER_FEATURE_SOURCE_DATA WHERE CUSTOMER_ID = 102;

ALTER ONLINE FEATURE TABLE CUSTOMER_FEATURES_ONLINE REFRESH;

SELECT * FROM CUSTOMER_FEATURES_ONLINE ORDER BY CUSTOMER_ID;
```

Do not directly update or delete serving rows. Change the governed source and refresh so offline and online definitions stay consistent. Suspend/resume controls background synchronization:

```sql
ALTER ONLINE FEATURE TABLE CUSTOMER_FEATURES_ONLINE
  SET TARGET_LAG = '1 minute';
ALTER ONLINE FEATURE TABLE CUSTOMER_FEATURES_ONLINE SUSPEND;
ALTER ONLINE FEATURE TABLE CUSTOMER_FEATURES_ONLINE RESUME;
```

After refresh completes, expect customer 101 with five orders, spend 22800.00, and risk 0.05; customer 102 is removed and customer 103 remains. Refresh completion can lag the request, so repeat the lookup when synchronization finishes.

## 5. Optional Snowflake ML Feature Store path

Use this in a **Snowsight Python notebook** with `snowflake-ml-python >= 1.41`. It registers an entity and feature view; `OnlineConfig` requests hybrid-table online serving by default. Obtain the active Snowpark session in the Python block below. Select the same database and `OnlineFeatures` schema in that notebook. Run the SQL privilege grants from your organization's admin setup first.

```python
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.use_schema("OnlineFeatures")
session.use_warehouse("SNOWFLAKE_LEARNING_WH")

from snowflake.ml.feature_store import (
    CreationMode, Entity, FeatureStore, FeatureView, OnlineConfig, StoreType
)

fs = FeatureStore(
    session=session,
    database=session.get_current_database(),
    name='ONLINEFEATURES',
    default_warehouse='SNOWFLAKE_LEARNING_WH',
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

customer = Entity(
    name='CUSTOMER',
    join_keys=['CUSTOMER_ID'],
    desc='Customer entity used for online lookup',
)
customer = fs.register_entity(customer)

feature_df = session.table('OnlineFeatures.CUSTOMER_FEATURE_SOURCE_DATA')
feature_view = FeatureView(
    name='CUSTOMER_BEHAVIOR',
    entities=[customer],
    feature_df=feature_df,
    timestamp_col='FEATURE_TS',
    refresh_freq='5 minutes',
    refresh_mode='INCREMENTAL',
    online_config=OnlineConfig(enable=True, target_lag='30 seconds'),
    desc='Customer order and risk features',
)

registered_fv = fs.register_feature_view(feature_view, version='v1')
fs.read_feature_view(
    feature_view=registered_fv,
    keys=[[101], [103]],
    feature_names=['ORDERS_30D', 'SPEND_30D', 'RISK_SCORE'],
    store_type=StoreType.ONLINE,
).show()
```

Feature Store is preferable when the team needs entity metadata, feature versions, point-in-time training sets, lineage, and a consistent offline/online contract. Source: [serving online features](https://docs.snowflake.com/en/developer-guide/snowflake-ml/feature-store/online-feature-store).

The Feature Store uses the existing `ONLINEFEATURES` schema here. If you changed the schema name, update both `use_schema` and the Feature Store `name`. This optional path creates additional registered feature objects independently of the SQL online table.

## 6. Review

1. Why is `CUSTOMER_ID` the online lookup key?
2. Why does the online table read from a view rather than directly from the source table?
3. Why must corrections be applied to the source?
4. What does the target lag describe?
5. When would entity metadata, feature versions, and training-set lineage justify the Feature Store API?

Use key lookups for the serving path. A source table remains the mutable input, and refreshing the serving copy does not replace upstream data-quality checks.

## 7. Optional cleanup

If you ran the Feature Store API extension, remove its registered feature view through the same API first. Run this Python block only in the session where `fs`, `registered_fv`, and `customer` were created:

```python
fs.delete_feature_view(registered_fv)
fs.delete_entity(customer.name)
```

Then remove the SQL lab objects in the original database:

```sql
USE SCHEMA OnlineFeatures;
USE WAREHOUSE SNOWFLAKE_LEARNING_WH;
DROP ONLINE FEATURE TABLE IF EXISTS CUSTOMER_FEATURES_ONLINE;
DROP VIEW IF EXISTS CUSTOMER_FEATURE_SOURCE;
DROP TABLE IF EXISTS CUSTOMER_FEATURE_SOURCE_DATA;
DROP SCHEMA IF EXISTS OnlineFeatures RESTRICT;
```

`RESTRICT` keeps the schema if Feature Store metadata or other objects remain. The database and learning warehouse are retained. Do not remove a schema that contains shared feature registrations.